Under construction!

# Step 4: Metadata Analysis Playground

Under construction!

This notebook provides a series of perspectives on the metadata, including general statistics and visualization. Results are stored in the output folder.

In [ ]:
# for the font issues, check 
# https://albertauyeung.github.io/2020/03/15/matplotlib-cjk-fonts.html/

In [ ]:
import json
import pandas as pd
from collections import Counter
import matplotlib.pyplot as plt
import numpy as np
import matplotlib.font_manager as fm
import seaborn as sns
import csv
import os
import yaml
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.cluster import KMeans
from sklearn.metrics.pairwise import cosine_similarity
import isodate
import pprint

plt.rcParams['font.family'] = 'Noto Sans CJK JP'  # or 'Arial', 'Times New Roman', etc.

In [ ]:
fonts = {'en': 'NotoSansJP-Light.otf', 'ja': 'NotoSansJP-Light.otf', 'ko': 'NotoSansKR-Light.otf', 'zh-Hant': 'NotoSansTC-Light.otf', 'zh-Hans': 'NotoSansSC-Light.otf'}

dataset_config = "./config/dataset_config.yml"
catfile = '*combined_cleaned_data.csv'
commentsfile = '_comments_cleaned_langdetect.csv'
metadatafile = '_metadata_cleaned_langinfo.csv'

output = './output/'

In [ ]:
# data screening

print(os.path.abspath(dataset_config))
print(os.path.getsize(dataset_config))
with open(dataset_config, "r") as f:
    config = yaml.safe_load(f)
# Expand paths relative to working dir
directories = {
    key: {
        'wd': (
            [os.path.join(os.getcwd(), path) for path in value['wd']]
            if value['wd'] != None
            else []
        ),
        'catdir': (
            os.path.join(os.getcwd(), value['catdir'])
            if value['catdir'] != None
            else ''
        ),
    }
    for key, value in config.items()
}

pprint.pprint(directories)

In [ ]:
relevantcols = ['videoId', 'videoSearchRegion', 'publishedAt', 'channelId', 
                'title', 'description', 'channelTitle', 'categoryId', 'duration' 
                'viewCount', 'likeCount', 'dislikeCount', 'commentCount']
langcols =['comp_verdict', 'num_languages', 'language_homogeneity']

# Helper

In [ ]:
def parseYouTubeDurationData(dur):
    if dur:
        return isodate.parse_duration(dur).total_seconds()
    else:
        return f'parsing error {dur}'

In [ ]:
def cleanup(d):
    tmp = d
    tmp['commentCount'] = tmp['commentCount'].replace('missing', np.nan)
    tmp['commentCount'] = tmp['commentCount'].astype('float64')
    tmp['likeCount'] = tmp['likeCount'].replace('missing', np.nan)
    tmp['likeCount'] = tmp['likeCount'].astype('float64')
    tmp['dislikeCount'] = tmp['dislikeCount'].replace('missing', np.nan)
    tmp['dislikeCount'] = tmp['dislikeCount'].astype('float64')
    tmp['duration'] = tmp['duration'].apply(parseYouTubeDurationData)
    return tmp

In [ ]:
#stats functions
def statistics(d):
    print(f'n = {len(d)} videos')
    print(f'viewcount for all videos = {d["viewCount"].sum()} views')
    print(f'like count for all videos = {d["likeCount"].sum()} likes')
    print(f'dislike count for all videos = {d["dislikeCount"].sum()} dislikes')
    print(f'comment count for all videos = {d["commentCount"].sum()} comments')
    print('stats complete')

In [ ]:
# Function to identify metadata files in a directory and create a list of their filenames
def identify_metadatafiles_in_directory(directory, type):
    file_list = []
    for file_name in os.listdir(directory):
        if file_name.endswith(type):
            file_list.append(file_name)
    return file_list

In [ ]:
tmpcols = ['num_languages', 'language_homogeneity']
def loadmetadatafiles(path, addlang=True):
    files = identify_metadatafiles_in_directory(path, metadatafile)
    print(files)
    result = {}
    for file in files:
        file_label = file.split(f'_{metadatafile}')[0]
        print('reading data for: ', file_label)
        #print(file)
        result[file_label] = pd.read_csv(os.path.join(path, file))
            #print('all columns in data: ', result[file_label].columns)
    return result

In [ ]:
def explode_videos(data):
    # Define which columns are multi-select (assumed to be stored as lists or comma-separated strings)
    multi_select_cols = [
        'Formal Elements (multiple possible)',
        'Values (Subjective Evaluation) muliple possible',
        'Content Focus (choose 0-2)',
        'Other Significant Tags (please describe new tags in the tag description column)',
        'Other Noteworthy Content Elements',
        'Other Noteworthy Formal Elements',
        'Other Noteworthy Evaluation',
    ]

    # One-hot encode multi-select fields
    for col in multi_select_cols:
        if col in data.columns:
            data[col] = data[col].fillna('').apply(lambda x: [i.strip() for i in x.split(',')] if isinstance(x, str) else x)
            mlb = MultiLabelBinarizer()
            transformed = pd.DataFrame(mlb.fit_transform(data[col]), columns=[f"{col}: {cls}" for cls in mlb.classes_])
            data = data.drop(columns=col).join(transformed)

    # One-hot encode single-select categorical fields
    single_select_cols = [
        'Main Video Source (choose 1)',
        'YouTube Shorts (yes no)',
        'Content Type (choose 0-1)'
    ]

    data = pd.get_dummies(data, columns=single_select_cols)
    print(type(data))
    return data

# Execute data loading and enhancement

In [ ]:
def filterdatatables(d):
    print('filtering data')
    result = {}
    for k in d.keys():
        print(k)
        try:
            type = k.split('_')[2][0:3]
            if 'vie' in type:
                type = 'view'
            elif 'rel' in type:
                type = 'rel'
            else: 
                type = 'n/a'
        except:
            print('type could not be detected')
            type = 'error'
        #print('columns in data: ', d[k].columns)
        result[k] = d[k].copy()
        result[k]["type"] = type
        result[k]["publishedAt"] = pd.to_datetime(result[k]['publishedAt'])        
 #       for key in d[k]:

 #           if 'viewCount' in d[k][key]['metadata']:
 #               viewC = d[key]['metadata']['viewCount']
 #           if 'likeCount' in d[key]['metadata']:
 #               likeC = d[key]['metadata']['likeCount']
 #           if 'commentCount' in d[key]['metadata']:
 #               commC = d[key]['metadata']['commentCount']
 #           df.loc[len(df)] = [key, type, d[key]['datetime'], viewC, likeC, commC]
    return result

## create metadata files for each game and enhance with cat-data   

In [ ]:
def ingest_metadata(game, explode=False):
    print(game)
    data = {}
    for d in directories[game]['wd']:
        #print(d)
        d_k = d.split('/')[-2]
        print('working on d:', d_k)
        data[d_k] = loadmetadatafiles(d, True)
        print(data[d_k].keys())
        print('metadata loaded successfully')
        data[d_k] = filterdatatables(data[d_k])
        print('metadata filtered successfully') 
        for k in data[d_k].keys():
            print('working on files in', k)
            print('columns in files', k)
            data[d_k][k] = cleanup(data[d_k][k])
            #print('data columns after cleanup', data[d_k][k].columns)
            print("Unique videoSearchRegion values in metadata:", data[d_k][k]['videoSearchRegion'].unique())
            print('data columns in metadata')
            print(data[d_k][k].columns)
            print(f"value counts after enrichment for {k}:")
            print(data[d_k][k]['Content Type (choose 0-1)'].value_counts(dropna=False))
            if explode:
                print('exploding data')
                data[d_k][k] = explode_videos(data[d_k][k])
            #data[d_k][k] = pd.DataFrame(data[d_k][k])
            print(f'length of dataset {k}: {len(data[d_k][k])}')
    return data

## execute data ingest

In [ ]:
datasets = {}
datasets_exploded = {}

for game in directories.keys():
    print(game)
    datasets[game] = {}
    datasets_exploded[game] = {}
    datasets[game] = ingest_metadata(game, False)
    datasets_exploded[game] = ingest_metadata(game, True)

# Overview Statistics

In [ ]:
# Create a list to collect stats
stats = []

# Iterate through the datasets and collect stats
for game in directories.keys():
    for dir in datasets[game]:
        date = dir[-6:]
        for l in datasets[game][dir]:
            print(l)
            lang = l.split('_')[2]
            print(lang)
            viewcount = datasets[game][dir][l]['viewCount'].sum()
            likecount = datasets[game][dir][l]['likeCount'].sum()
            commentcount = datasets[game][dir][l]['commentCount'].sum()
            durationtotal = datasets[game][dir][l]['duration'].sum()
            durationmean = datasets[game][dir][l]['duration'].mean()
            stats.append({
                'title': game,
                'date': date,
                'language': lang,
                'viewCount': viewcount,
                'likeCount': likecount,
                'commentCount': commentcount,
                'durationTotal': durationtotal,
                'durationMean': durationmean
            })

# Convert the list of stats to a DataFrame
stats_df = pd.DataFrame(stats)

# Save to CSV
stats_df.to_csv(os.path.join(output, 'ytma_metadataanalysis_basicstats.csv'), index=False)

In [ ]:
def group_duration(seconds):
    if seconds <= 60:
        return '1. ≤ 1 minute'
    elif seconds <= 300:
        return '2. > 1 ≤ 5 minutes'
    elif seconds <= 600:
        return '3. > 5 ≤ 10 minutes'
    elif seconds <= 1800:
        return '4. > 10 minutes ≤ 30 minutes'
    else:
        return '5. > 30 minutes'

# Example applied to a DataFrame
group_counts = {}
for game in directories.keys():
    for dir in datasets[game]:
        for l in datasets[game][dir]:
            datasets[game][dir][l]['duration_group'] =  datasets[game][dir][l]['duration'].apply(group_duration)
            group_counts[l] = datasets[game][dir][l]['duration_group'].value_counts().sort_index()
            group_counts[l].plot(kind='bar', color='skyblue', edgecolor='black')
            plt.title(f'Video Duration Group Distribution: {l}')
            plt.xlabel('Duration Group')
            plt.ylabel('Number of Videos')
            plt.xticks(rotation=45, ha='right')
            plt.tight_layout()
            plt.show()
df = pd.DataFrame(group_counts)
df.to_csv(os.path.join(output, 'ytma_metadataanalysis_durationgroups.csv'))

In [ ]:
for game in directories.keys():
    for dir in datasets[game]:
        for l in datasets[game][dir]:
            plt.bar(datasets[game][dir][l]['videoId'], datasets[game][dir][l]['duration'])
            plt.xticks(rotation=45, ha='right')
            plt.show()

In [ ]:
def plot_mostcommon(common, font):
    fprop = fm.FontProperties(fname='./fonts/'+fonts[font])
    x, y = zip(*common)
    plt.yticks(np.arange(len(x)), x, fontproperties=fprop, fontsize=8, rotation=0)
    plt.xticks(np.arange(common[0][1]+2), fontproperties=fprop, fontsize=8, rotation=0)
    plt.title(font, fontproperties=fprop, fontsize=18)
    plt.barh(x, y)
    plt.show()

'videoId', 'videoLanguage', 'publishedAt', 'channelId',
       'title', 'description', 'channelTitle', 'tags', 'categoryId',
       'liveBroadcastContent', 'defaultLanguage', 'duration', 'dimension',
       'definition', 'caption', 'licensedContent', 'contentRating',
       'projection', 'uploadStatus', 'privacyStatus', 'license', 'embeddable',
       'publicStatsViewable', 'madeForKids', 'viewCount', 'likeCount',
       'dislikeCount', 'favoriteCount', 'commentCount', 'topicCategories'

In [ ]:
def analyzedatatables(data):
    print("analyzedatatables")
    print(data.keys())
    for k in data.keys():
        print(k)
        language = k.split("_")[2]
        print(language)
        print('commentCount unique')
        print([float(i) for i in data[k]['commentCount'].unique()])
        print('statistics')
        statistics(data[k])
        print('channels info')
        print(len(set(data[k]['channelTitle'])))
        channelcount = Counter()
        channelcount.update(data[k]['channelTitle'])
        print(channelcount.most_common())
        plot_mostcommon(channelcount.most_common(20), language)
    return data

In [ ]:
def showpublicationtimeline(data):
    for k in data.keys():
        #fprop = fm.FontProperties(fname='./fonts/'+fonts[font])
        #x, y = zip(*common)
        #plt.yticks(np.arange(len(x)), x, fontproperties=fprop, fontsize=8, rotation=0)
        #plt.xticks(np.arange(common[0][1]+2), fontproperties=fprop, fontsize=8, rotation=0)
        #plt.title(font, fontproperties=fprop, fontsize=18)
        #plt.barh(x, y)
        #tmp.index = tmp.index.map('_'.join)
        #tmp = data[k]["publishedAt"].groupby([data[k]["publishedAt"].dt.year, data[k]['publishedAt'].dt.month]).size()
        data[k]['year'] = data[k]["publishedAt"].dt.year
        data[k]['month'] = data[k]["publishedAt"].dt.month
        tmp = data[k][["videoId", "year", "month"]].groupby([data[k]["year"], data[k]["month"]]).size().reset_index()
        #tmp = data[k]["publishedAt"].groupby([data[k]["publishedAt"].dt.year, data[k]['publishedAt'].dt.month]).aggregate(lambda x: ','.join(map(str, x)))
        tmp["date"] = tmp['year'].astype(str) + "-" + tmp['month'].astype(str)
        result = Counter(pd.Series(tmp[0].values,index=tmp["date"]).to_dict())
        #result = Counter(dict(tmp["date"], tmp[0]))
        print(result)
        plt.bar(result.keys(), result.values())
        plt.xticks(rotation=45, ha='right')
        plt.show()
    return True

In [ ]:
def fulldataanalysis(data):
    analyzedatatables(data)
    showpublicationtimeline(data)
    return True

In [ ]:
# skipping for now...
for k in datasets.keys():
    print('game: ', k)
    for d in datasets[k]:
        print('analyzing data in folder: ', d)
        fulldataanalysis(datasets[k][d])

# language analysis

In [ ]:
# average number of languages in data:
for k in datasets:
    print('game: ', k)
    for d in datasets[k]:
        print('analyzing language data in dataset: ', d)
        for l in datasets[k][d]:
            print(l)
            print(datasets[k][d][l]['num_languages'].mean())

## show heatmaps of content types and focus

In [ ]:
# 'Content Type (choose 0-1)', 'Content Focus (choose 0-2)',        'Other Noteworthy Content Elements',       'Values (Subjective Evaluation) muliple possible',
def showcrosstabasheatmap(data, a, b, data_label):
    ct = pd.crosstab(data[a], data[b])
    print(ct)

    # Plot heatmap
    sns.heatmap(ct, annot=True, fmt='d', cmap='YlGnBu')
    plt.title(f'{a} vs {b} Heatmap for {data_label}')
    plt.ylabel(a)
    plt.xlabel(b)
    plt.show()

for k in datasets:
    print('game: ', k)
    for d in datasets[k]:
        print('analyzing language data in dataset: ', d)
        for l in datasets[k][d]:
            print(l)
            showcrosstabasheatmap(datasets[k][d][l], 'Content Type (choose 0-1)', 'num_languages', l)
            #print(datasets[k][d][l]['Content Type (choose 0-1)'].value_counts())
            #print(datasets[k][d][l]['num_languages'].value_counts())
            

## show heatmaps of language data

In [ ]:
for k in datasets:
    print('game: ', k)
    for d in datasets[k]:
        print('analyzing language data in dataset: ', d)
        for l in datasets[k][d]:
            print(l)
            try:
                showcrosstabasheatmap(datasets[k][d][l], 'Values (Subjective Evaluation) muliple possible', 'num_languages', l)
            except:
                print('data could not be displayed, check contents')

# Simple comparison

In [ ]:
def flatten_metadata(nested_dict):
    dfs = []
    for outer_key, inner_dict in nested_dict.items():
        print(outer_key)
        print(inner_dict)
        if isinstance(inner_dict, pd.DataFrame):
            print('yes') 
            print(inner_dict.head())
            dfs.append(inner_dict)
        else:
                print(f"Skipping non-DataFrame at {outer_key} -> {inner_key}: type={type(maybe_df)}")
    if dfs:
        return pd.concat(dfs, ignore_index=True)
    else:
        raise ValueError("No DataFrames found in the nested structure.")
    
def flatten_metadata2(data_dict):
    dfs = []
    for key, maybe_df in data_dict.items():
        if isinstance(maybe_df, pd.DataFrame):
            dfs.append(maybe_df)
        else:
            print(f"Skipping non-DataFrame at {key}: type={type(maybe_df)}")
    if dfs:
        return pd.concat(dfs, ignore_index=True)
    else:
        raise ValueError("No DataFrames found in the structure.")

In [ ]:
def language_comparison_all2(data):
    if 'relevanceLanguage' not in data.columns:
        raise ValueError("'lang' column is missing from DataFrame")

    numeric_cols = [column for column in data.select_dtypes(include='number').columns if column not in ['categoryId', 'viewCount', 'likeCount', 'dislikeCount', 'commentCount']]
    print("Numeric columns used:", numeric_cols)

    lang_summary = data.groupby('relevanceLanguage')[numeric_cols].mean().T
    print("Language summary shape:", lang_summary.shape)

    if lang_summary.empty:
        print("Warning: Language summary is empty — no data to plot.")
        return

    plt.figure(figsize=(12, 10))
    sns.heatmap(lang_summary, annot=True,fmt='.2f', cmap='mako', cbar=True)
    plt.title("Tag Feature Prevalence by Language")
    plt.xlabel("Language")
    plt.ylabel("Feature")
    plt.tight_layout()
    plt.show()


for k in datasets:
    print('game: ', k)
    for d in datasets[k]:
        print('analyzing data in folder: ', d)
        flat_metadata = flatten_metadata2(datasets[k][d])
        print(flat_metadata.relevanceLanguage.unique())
        #print(flat_metadata.select_dtypes(include='number').head())
        print('counting', flat_metadata['relevanceLanguage'].value_counts())
        language_comparison_all2(flat_metadata)

In [ ]:
def language_comparison_single (data, feat):
    # Group by language
    data.groupby('relevanceLanguage')[feat].mean().plot(kind='bar', color='steelblue')
    plt.title(f"Usage of '{feature}' by Language")
    plt.ylabel("Proportion")
    plt.xlabel("Language")
    plt.ylim(0, 1)
    plt.grid(axis='y')
    plt.tight_layout()
    plt.show()

feature = 'Content Type (choose 0-1)_reactionvideo'

for k in datasets:
    print('game: ', k)
    for d in datasets[k]:
        print('analyzing data in folder: ', d)
        flat_metadata = flatten_metadata2(datasets[k][d])
        language_comparison_single(flat_metadata, feature)

In [ ]:
def normalize_cell(cell):
    # If the cell is a list (like ["avatar; text"]), join into one string
    #print(cell)
    if isinstance(cell, list):
        cell = '; '.join(str(item) for item in cell if isinstance(item, str))
    elif not isinstance(cell, list):
        cell = str(cell) if cell is not None else "None"
    
    return cell.strip()

In [ ]:
def compare_exploded_categories(data, cat):
    print(repr(data.columns.tolist()))
    df_clean = data.dropna(subset=[cat]).copy()
    cat_label = f'{cat}_list'
    df_clean[cat_label] = df_clean[cat].apply(normalize_cell)

    df_exploded = df_clean.explode(cat_label)
    df_exploded = df_exploded[df_exploded[cat_label] != '']
    
    print("Unique exploded tags:", df_exploded[cat_label].unique())
    print("Exploded shape:", df_exploded.shape)

    tag_counts = (
        df_exploded
        .groupby(['lang', cat_label])
        .size()
        .unstack(fill_value=0)
    )
    tag_props = tag_counts.div(tag_counts.sum(axis=1), axis=0)

    if not tag_props.empty:
        plt.figure(figsize=(12, 6))
        sns.heatmap(tag_props, annot=True, cmap='mako', cbar=True)
        plt.title(f"Proportion of {cat} by Language")
        plt.xlabel(cat)
        plt.ylabel("Language")
        plt.tight_layout()
        plt.show()
    else:
        print("No data available to plot.")

category = 'Content Type (choose 0-1)'
compare_exploded_categories(test, category)

In [ ]:
def compare_exploded_categories_separately(data, cat):
    df_clean = data.dropna(subset=[cat]).copy()
    df_clean['normalized_tags'] = df_clean[cat].apply(normalize_cell)

    # Split tags into list of individual tags
    df_clean['tag_list'] = df_clean['normalized_tags'].str.split(';')
    #print(df_clean['tag_list'])

    # Strip spaces from individual tags
    df_clean['tag_list'] = df_clean['tag_list'].apply(lambda tags: [t.strip() for t in tags if t.strip()])
    #print(df_clean[['lang', 'normalized_tags', 'tag_list']].head(10))

    # Explode into one row per tag
    data_exploded = df_clean.explode('tag_list')
    # Count occurrences of each tag per language
    tag_counts = data_exploded.groupby(['lang', 'tag_list']).size().unstack(fill_value=0)
    print(tag_counts)

    # If you want to see which tags are present in each language
    tag_presence = tag_counts.astype(bool).astype(int)
    print(tag_presence)
    tags_by_language = (
    data_exploded.groupby('lang')['tag_list']
    .apply(lambda x: set(x.dropna()))
    )
    print(tags_by_language)

    tag_props = tag_counts.div(tag_counts.sum(axis=1), axis=0)
    print(tag_props)
    # Counts heatmap
    plt.figure(figsize=(10, 6))
    sns.heatmap(tag_counts, annot=True, fmt='d', cmap='Blues')
    plt.title("Tag Counts by Language")
    plt.show()

    # Proportions heatmap
    plt.figure(figsize=(10, 6))
    sns.heatmap(tag_props, annot=True, fmt=".2f", cmap='Greens')
    plt.title("Tag Proportions by Language")
    plt.show()

In [ ]:

category = 'Formal Elements (multiple possible)'
category = 'Values (Subjective Evaluation) muliple possible'
compare_exploded_categories_separately(test, category)

In [ ]:
category = 'Formal Elements (multiple possible)'
#category = 'Values (Subjective Evaluation) muliple possible'
compare_exploded_categories_separately(test, category)

In [ ]:
#category = 'Formal Elements (multiple possible)'
#category = 'Values (Subjective Evaluation) muliple possible'
category = 'Content Type (choose 0-1)'
compare_exploded_categories_separately(test, category)

# Cluster Analysis and Visualization

In [ ]:
# Define non-feature columns globally to reuse
non_feature_cols = ['videoId', 'relevanceLanguage', 'link', 'Content match (yes no)',
                    'Language Match (yes no)', 'new tags and other memos', 'lang', 'source_file']

def get_selected_columns(data, cluster_columns):
    if cluster_columns:
        return [col for col in data.columns if col.startswith(cluster_columns)]
    return [col for col in data.columns if col not in non_feature_cols]

def clustering_videos(data, cluster_columns=None):
    sel_cols = get_selected_columns(data, cluster_columns)
    X = data[sel_cols].copy()
    kmeans = KMeans(n_clusters=5, random_state=0)
    data['cluster'] = kmeans.fit_predict(X)
    return data

def similaritymatrix_videos(data, cluster_columns=None):
    sel_cols = get_selected_columns(data, cluster_columns)
    X = data[sel_cols].copy()
    similarity_matrix = cosine_similarity(X)
    return similarity_matrix

def discoverpatterns_videos(data, cluster_columns=None):
    sel_cols = get_selected_columns(data, cluster_columns)
    return data[sel_cols].corr()

def interpretclusters_videos(data, cluster_columns=None):
    sel_cols = get_selected_columns(data, cluster_columns)
    X = data[sel_cols + ['cluster']].copy()
    interpret_clusters = X.groupby('cluster').mean().T
    interpret_clusters = interpret_clusters.sort_index().round(2)
    return interpret_clusters

def interpretclusters_videos_perlanguage(data, cluster_columns=None):
    sel_cols = get_selected_columns(data, cluster_columns)
    X = data[sel_cols + ['cluster', 'lang']].copy()
    X['cluster'] = X['cluster'].astype(int)
    numeric_cols = X.select_dtypes(include='number').columns.difference(['cluster'])
    grouped = X.groupby(['lang', 'cluster'])[numeric_cols].mean().round(2)
    return grouped


In [ ]:
cluster_columns = ('Content Type (choose 0-1)', 
           'Values (Subjective Evaluation) muliple possible', 
           'Content Focus (choose 0-2)')
clustered_data = clustering_videos(processed_test, cluster_columns)
print(clustered_data['cluster'].value_counts())
sim_matrix = similaritymatrix_videos(clustered_data, cluster_columns)
print(sim_matrix)
patterns = discoverpatterns_videos(clustered_data, cluster_columns)
print(patterns)
cluster_summary = interpretclusters_videos(clustered_data, cluster_columns)
print(cluster_summary)
lang_summary = interpretclusters_videos_perlanguage(clustered_data, cluster_columns)
print(lang_summary)


## visualizations

In [ ]:
def tagfrequenciesforsinglelanguage(data, l):
    # All tag frequencies for a single language across clusters
    subset = data.loc[l]

    plt.figure(figsize=(12, 10))
    sns.heatmap(subset.T, annot=False, cmap='coolwarm', center=0.5)
    plt.title(f"Tag Prevalence by Cluster for Language: {l}")
    plt.xlabel("Cluster")
    plt.ylabel("Tag Feature")
    plt.tight_layout()
    plt.show()

tagfrequenciesforsinglelanguage(lang_summary, 'jp')

def visualizedistributionforsinglefeatureacrosslanguages(data, feat):
# Choose a feature to visualize

    # Ensure 'cluster' is an integer for grouping
    data['cluster'] = data['cluster'].astype(int)

    # Group by language and cluster, take mean of the selected feature
    grouped = data.groupby(['lang', 'cluster'])[feat].mean().unstack('cluster')

    # Plot the heatmap
    plt.figure(figsize=(10, 6))
    sns.heatmap(grouped, annot=True, cmap='viridis')
    plt.title(f"Distribution of '{feat}' by Language and Cluster")
    plt.xlabel("Cluster")
    plt.ylabel("Language")
    plt.tight_layout()
    plt.show()

feature = 'Content Type (choose 0-1)_reactionvideo'

visualizedistributionforsinglefeatureacrosslanguages(clustered_data, feature)

def visualize_features_by_language_and_cluster(data, features):
    # Ensure cluster column is in the correct format
    data['cluster'] = data['cluster'].astype(int)

    for feat in features:
        if feat not in data.columns:
            print(f"Feature '{feat}' not found in data. Skipping.")
            continue

        # Group by language and cluster, then compute mean
        grouped = data.groupby(['lang', 'cluster'])[feat].mean().unstack('cluster')

        # Plot the heatmap
        plt.figure(figsize=(10, 6))
        sns.heatmap(grouped, annot=True, cmap='viridis', fmt=".2f")
        plt.title(f"Distribution of '{feat}' by Language and Cluster")
        plt.xlabel("Cluster")
        plt.ylabel("Language")
        plt.tight_layout()
        plt.show()

features = [col for col in clustered_data.columns if col.startswith('Content Type')]

visualize_features_by_language_and_cluster(clustered_data, features)

def plot_cluster_distribution(data):
    data['cluster'] = data['cluster'].astype(str)  # Convert to string for plotting
    plt.figure(figsize=(10, 6))
    sns.countplot(data=data, x='lang', hue='cluster', palette='tab10')
    plt.title('Cluster Distribution per Language')
    plt.xlabel('Language')
    plt.ylabel('Number of Videos')
    plt.legend(title='Cluster')
    plt.tight_layout()
    plt.show()

plot_cluster_distribution(clustered_data)